In [ ]:
import json
import pandas as pd

# 1. โหลดข้อมูลสถิติการเลือกตั้ง (stats_cons.json) และข้อมูลจังหวัด (info_province.json)
with open('stats_cons.json', 'r', encoding='utf-8') as f:
    stats_data = json.load(f)

with open('info_province.json', 'r', encoding='utf-8') as f:
    province_info = json.load(f)

# 2. สร้าง Mapping รหัสจังหวัดเป็นชื่อจังหวัด เพื่อการแสดงผลที่เข้าใจง่าย
prov_map = {p['prov_id']: p['province'] for p in province_info['province']}

# 3. ประมวลผลข้อมูลรายจังหวัด
province_comparison = []

for prov in stats_data['result_province']:
    p_id = prov['prov_id']
    p_name = prov_map.get(p_id, p_id)

    # ดึงค่าผู้มาใช้สิทธิของทั้งสองระบบ
    turn_out_mp = prov.get('turn_out', 0)
    turn_out_party = prov.get('party_list_turn_out', 0)

    # คำนวณผลต่าง
    diff = turn_out_mp - turn_out_party
    abs_diff = abs(diff) # ผลต่างสัมบูรณ์ (จำนวนคนที่ไม่เท่ากัน)

    province_comparison.append({
        'จังหวัด': p_name,
        'มาใช้สิทธิ (ส.ส.เขต)': turn_out_mp,
        'มาใช้สิทธิ (บัญชีรายชื่อ)': turn_out_party,
        'ผลต่าง (คน)': diff,
        'ผลต่างสัมบูรณ์ (คน)': abs_diff
    })

# 4. แปลงเป็น DataFrame และจัดเรียงข้อมูลตามผลต่างสัมบูรณ์จากมากไปน้อย
df = pd.DataFrame(province_comparison)
df_sorted = df.sort_values(by='ผลต่างสัมบูรณ์ (คน)', ascending=False)

# 5. แสดงผลลัพธ์
print("ตารางเปรียบเทียบผลต่างจำนวนผู้มาใช้สิทธิแยกตามจังหวัด (Top 15):")
print(df_sorted.head(15).to_string(index=False))

# 6. บันทึกผลลัพธ์เป็นไฟล์ CSV
df_sorted.to_csv('province_turnout_comparison.csv', index=False, encoding='utf-8-sig')